# PlantDoc + YOLOv8 - experimentos do TCC
Ative uma GPU em **Runtime > Change runtime type**. Use uma exportação YOLOv8 do PlantDoc que contenha `train`, `valid`, `test` e `data.yaml`.

In [ ]:
!git clone https://github.com/SoulStorm0/plantdoc-yolov8-tcc.git
%cd plantdoc-yolov8-tcc
!python -m pip install -e .

## Preparação reproduzível do dataset
A fonte oficial usa nomes incompatíveis com Windows. O conversor lê os blobs Git, converte Pascal VOC para YOLO e cria o split 70/20/10 com 27 classes suportadas.

In [ ]:
!git clone --depth 1 --no-checkout https://github.com/pratikkayal/PlantDoc-Object-Detection-Dataset.git /content/plantdoc_official
!python scripts/prepare_official_plantdoc.py --repo /content/plantdoc_official --output /content/plantdoc_yolo_27 --min-class-instances 20
from pathlib import Path
data_yaml = Path('/content/plantdoc_yolo_27/data.yaml')
print(data_yaml.read_text())

In [ ]:
!python -m plantdoc_tcc audit --data "{data_yaml}" --expected-classes 27

## Treinamento
Comece por uma estratégia e pela triagem de 100 épocas. O grid integral é grande; promova para 200/300 épocas apenas as melhores configurações na validação. Nunca escolha hiperparâmetros pelo teste.

In [ ]:
!python -m plantdoc_tcc train --data "{data_yaml}" --config configs/experiments.json --strategy baseline --epoch 100 --device 0

Repita com `--strategy class_weighted` e `--strategy focal`. Depois de escolher o melhor modelo somente pela validação, execute a avaliação final uma vez:

In [ ]:
BEST = '/content/plantdoc-yolov8-tcc/runs/plantdoc/SEU_RUN/weights/best.pt'
!python -m plantdoc_tcc evaluate --weights "{BEST}" --data "{data_yaml}" --split test --output artifacts/test_metrics.json